# Day 14 — Tokens/Sec vs Batch Size (mini-project)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day14-tokens-sec-vs-batch-size.ipynb)

**Goal:** measure the throughput-vs-batch-size curve on your own hardware, then explain its shape with the roofline and the KV byte-ledger from Days 8–13.
~45 min on a free T4; CPU also works (slower, same shape).

In [ ]:
# One and only pip install cell
!pip install -q transformers accelerate matplotlib
# Expected output: Successfully installed transformers-... accelerate-... matplotlib-...

## 1. Specs: your hardware, your model

T4: 300 GB/s HBM bandwidth, 65 TFLOP/s fp16 dense, 16 GB HBM.
SmolLM2-135M: 135M params, 30 layers, GQA (3 KV heads x 64 head-dim). fp16 weights = 269 MB.
We read the KV geometry from `model.config` below — never hardcode GQA numbers.

In [ ]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)  # Expected on Colab T4: device: cuda
tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-135M", torch_dtype=torch.float16).to(device).eval()
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params:,}")  # Expected: params: ~134,515,008
c = model.config
kv_per_token = 2 * c.num_hidden_layers * c.num_key_value_heads * (c.hidden_size // c.num_attention_heads) * 2
print(f"KV bytes/token: {kv_per_token:,} (~{kv_per_token/1024:.1f} KiB)")  # Expected: ~23,040 (~22.5 KiB)
w_bytes = n_params * 2
print(f"fp16 weight bytes: {w_bytes/1e6:.1f} MB")  # Expected: ~269.0 MB

## 2. Predict first: the roofline-plus-KV curve

Before timing anything, write down the prediction. Per decode step:
`bytes/step = W + B x S x k` (weights + every sequence's KV re-read).
`tok/s = B x BW / bytes_per_step`. The bends: KV-traffic crossover `B x S x k = W`,
HBM cap `(16 - W) GB / (S x k)`. There is NO compute knee at this context: honest
intensity is `I(B) = B(W+attn) / (W + B*S*k)`, which asymptotes at `(W+attn)/(S*k)` ≈
22 FLOP/byte — ~10x below the 217 FLOP/byte ridge (batching grows KV traffic exactly
as fast as FLOPs, so intensity saturates instead of climbing).

In [ ]:
BW = 300e9          # T4 HBM bandwidth, bytes/s
PEAK = 65e12        # T4 fp16 dense TFLOP/s
VRAM = 16e9
PROMPT, GEN = 512, 128
S = PROMPT + GEN      # max context during the run

def predict(batch):
    step_bytes = w_bytes + batch * S * kv_per_token
    return batch * BW / step_bytes

batches = [1, 2, 4, 8, 16]
pred = {b: predict(b) for b in batches}
for b in batches:
    print(f"B={b:2d}  predicted {pred[b]:8.0f} tok/s   ratio-to-ideal {pred[b]/(b*pred[1]):.2f}")
# Expected: B= 1 predicted ~1057 tok/s ratio 1.00; B= 8 -> ~6202, ratio ~0.73; B=16 -> ~9506, ratio ~0.56
b_cross = w_bytes / (S * kv_per_token)
b_cap = (VRAM - w_bytes) / (S * kv_per_token)
attn_per_token = 4 * c.num_hidden_layers * S * c.hidden_size   # Day 9's corrected attention FLOPs
i_asymptote = (w_bytes + attn_per_token) / (S * kv_per_token)  # intensity as B -> infinity
ridge = PEAK / BW
print(f"\nbends: KV-traffic crossover B~{b_cross:.0f}, HBM cap B~{b_cap:.0f}")
print(f"no compute knee: intensity asymptotes at ~{i_asymptote:.0f} FLOP/byte vs ridge ~{ridge:.0f}")
# Expected: bends: KV-traffic crossover B~18, HBM cap B~1067
# Expected: no compute knee: intensity asymptotes at ~22 FLOP/byte vs ridge ~217

## 3. The benchmark harness

`bench(batch)`: one warmup, then 3 timed runs of 128 new tokens with the KV cache on.
Prompts are identical 512-token blocks so prefill cost is constant across batches.

In [ ]:
prompt = " ".join(["The memory hierarchy decides the speed of inference."] * 64)
ids = tok(prompt, return_tensors="pt")["input_ids"][:, :PROMPT]
print("prompt tokens:", ids.shape[1])  # Expected: prompt tokens: 512

def bench(batch, new_tokens=GEN, repeats=3):
    x = ids.repeat(batch, 1).to(device)
    attn = torch.ones_like(x)
    with torch.inference_mode():
        model.generate(x, attention_mask=attn, max_new_tokens=8, use_cache=True)  # warmup
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        with torch.inference_mode():
            model.generate(x, attention_mask=attn, max_new_tokens=new_tokens,
                           use_cache=True, pad_token_id=tok.eos_token_id)
        ts.append(new_tokens * batch / (time.perf_counter() - t0))
    return sum(ts) / len(ts)

r1 = bench(1)
print(f"smoke test B=1: {r1:.0f} tok/s")  # Expected on T4: 300-600 tok/s (Python overhead on a 135M model)

## 4. Measure the curve

This is the mini-project's core number. ~2-4 min on a T4.

In [ ]:
measured = {}
for b in batches:
    measured[b] = bench(b)
    print(f"B={b:2d}  measured {measured[b]:8.0f} tok/s   vs predicted {pred[b]:8.0f}  (ratio {measured[b]/pred[b]:.2f})")
# Expected: ratio-to-prediction roughly constant across B (same overhead factor);
# measured curve bends like the prediction: near-linear to B=8, drooping by B=16

## 5. Plot: measured vs predicted, with the three bends marked

In [ ]:
import matplotlib.pyplot as plt

bs = batches
plt.figure(figsize=(8, 5))
plt.plot(bs, [pred[b] for b in bs], "o--", label="predicted (roofline + KV ledger)")
plt.plot(bs, [measured[b] for b in bs], "s-", label="measured")
for bx, name, col in [(b_cross, "KV-traffic crossover", "red"),
                      (b_cap, "HBM cap", "green")]:
    plt.axvline(bx, color=col, linestyle=":", alpha=0.7)
    plt.text(bx, plt.ylim()[1] * 0.95, f" {name}\n B~{bx:.0f}", color=col, fontsize=8)
plt.text(0.02, 0.02, f"no compute knee: intensity asymptotes ~{i_asymptote:.0f} FLOP/byte\n"
         f"vs {ridge:.0f} ridge (unreachable at S={S})",
         transform=plt.gca().transAxes, fontsize=8, color="#6b7280",
         bbox=dict(boxstyle="round", fc="white", alpha=0.8))
plt.xscale("log"); plt.xlabel("batch size (log)"); plt.ylabel("tokens/sec")
plt.title("Day 14: tok/s vs batch — SmolLM2-135M on T4")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()
# Expected: measured hugs the predicted curve; the bend sits at the red crossover line,
# far left of the green HBM-cap line. No ridge line is drawn — at 640-token context the
# 217 FLOP/byte ridge is unreachable (intensity asymptotes at ~22 FLOP/byte).

## 6. Reconcile: where did the overhead go?

The measured curve should sit *below* prediction by a roughly constant factor (kernel launch
+ Python overhead — Day 11's profiler story). If one batch size is an outlier, re-run it.

In [ ]:
ratios = [measured[b] / pred[b] for b in batches]
for b, r in zip(batches, ratios):
    print(f"B={b:2d}  measured/predicted = {r:.2f}")
print(f"\nmean overhead factor: {sum(ratios)/len(ratios):.2f} "
      f"(spread {max(ratios)-min(ratios):.2f} — small spread = the ledger explains the shape)")
# Expected: ratios ~0.3-0.6 (overhead-dominated small model), spread < 0.15

## 7. The 8B/H100 ordering exercise (no GPU needed)

Same byte ledger, big-league numbers: W=16.06 GB, BW=3.35 TB/s, peak=989 TFLOP/s,
k=128 KiB/token, S=2560. Two bends plus one mirage: which bends exist, and which line
is never crossed? (Day 14 checkpoint 1.)

In [ ]:
W, BW8, PEAK8, k8, S8, VRAM8 = 16.06e9, 3.35e12, 989e12, 128*1024, 2560, 80e9
cross8 = W / (S8 * k8)
cap8 = (VRAM8 - W - 2e9) / (S8 * k8)
attn8 = 4 * 32 * S8 * 4096                       # Day 9's corrected attention FLOPs per token
i_asym8 = (W + attn8) / (S8 * k8)                 # intensity as B -> infinity
ridge8 = PEAK8 / BW8
s_thresh8 = W / (ridge8 * k8 - 4 * 32 * 4096)     # S below which the ridge is reachable
print(f"KV-traffic crossover: B~{cross8:.0f}")   # Expected: ~48
print(f"HBM cap:              B~{cap8:.0f}")     # Expected: ~185
print(f"intensity asymptote:  ~{i_asym8:.0f} FLOP/byte vs ridge ~{ridge8:.0f}: no compute knee")
print("Halve S: crossover ~96, cap ~370, asymptote ~100 — still 3x below the ridge.")
print(f"The ridge needs S < ~{s_thresh8:.0f} tokens: short-context decode can go compute-bound; 2.5k-ctx decode cannot.")
# Expected: the packet's headline numbers, recomputed from the honest intensity formula

## Wrap-up

You measured the serving curve and reconciled it against a byte ledger — the exact move
every optimization from here gets: predict, measure, reconcile the gap.

**Mini-project deliverable:** write `week02/mini-project.md` — setup (1 paragraph), the plot
from cell 5, five bullets explaining the shape, and "what I'd change with 8xH100"
(tensor parallelism: split weights so each GPU reads 1/8th — 8x bandwidth, 8x the slope).
Self-grade the Day 9-13 checkpoints cold, then tag the repo `week2-complete`.

Tomorrow (Day 15): the *how* of serving — a real `/v1/chat/completions` FastAPI server with SSE streaming.